# Is there a maintenance term? Hunter-Reiner versus Buzzi-Ferraris-Forzatti when parameters are uncertain

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/zuhairblr/middoe/blob/main/tests/tutorials/05_design_discrimination/b_hr_vs_bff/tutorial.ipynb)

**Two discrimination criteria** (`mbdoe_md` with `HR` and `BFF`).

Does the yeast spend substrate on maintenance (**MI**, with a maintenance coefficient `m`) or not
(**MIV**, plain Monod)? After a short preliminary campaign the parameters of both models are still
uncertain, and that matters for the design:

* **HR**, Hunter-Reiner, maximises the difference between the predictions and ignores how uncertain
  they are;
* **BFF**, Buzzi-Ferraris-Forzatti, divides that difference by the prediction uncertainty
  (measurement noise plus parameter covariance), so it avoids conditions where the models differ
  only because their parameters are poorly known.

You will design one run with each criterion and run the BFF one.

In [ ]:
# Google Colab only: install MIDDoE and fetch the model file(s); does nothing elsewhere
import sys
if 'google.colab' in sys.modules:
    import subprocess, urllib.request
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'git+https://github.com/zuhairblr/middoe.git'], check=True)
    for f in ['MI.py', 'MIV.py']:
        urllib.request.urlretrieve('https://raw.githubusercontent.com/zuhairblr/middoe/main/tests/tutorials/05_design_discrimination/b_hr_vs_bff/' + f, f)

In [ ]:
from pathlib import Path
import shutil
import numpy as np
import pandas as pd

from middoe.krnl_expera import expera
from middoe.iden_parmest import parmest
from middoe.iden_uncert import uncert
from middoe.log_utils import save_rounds
from middoe.des_md import mbdoe_md

QUICK = False   # True: tiny budgets and fewer designs, a check that everything runs (minutes), not a result.
HERE = Path.cwd()
# Start clean: expera() ADDS sheets to an existing data.xlsx, so remove what an earlier run wrote.
for old in ['data.xlsx', 'data_plots', 'modelling', 'confidence', 'post_processing', 'estimability',
            'meshindep_plots', 'design_plots', 'iden_results.jac']:
    p = HERE / old
    shutil.rmtree(p) if p.is_dir() else p.unlink(missing_ok=True)
np.random.seed(0)                                # same noise at every run

In [ ]:
theta = {'MI':  [0.25, 0.25, 0.88, 0.09],     # mu_max, Ks, Yxs, m   (the 'real' process here)
         'MIV': [0.15270, 0.40575, 0.47529]}    # mu_max, Ks, Yxs      (no maintenance)
models = {
    'can_m': ['MI', 'MIV'],
    'krt': {m: 'pys' for m in theta},
    'src': {m: str(HERE / f'{m}.py') for m in theta},
    'theta': {m: [x * 1.2 for x in v] for m, v in theta.items()},
    't_l': {m: [x * 0.5 for x in v] for m, v in theta.items()},
    't_u': {m: [x * 1.5 for x in v] for m, v in theta.items()},
}
insilicos = {'tr_m': 'MI', 'theta': theta['MI'], 'errt': 'rel',
             'prels': {'1': {'u1': 0.05, 'u2': 30, 'y10': 5.5}}}   # a single preliminary run

system = {
    'tvi': {'u1': {'stps': 5, 'cvp': 'CPF', 'const': 'rel', 'min': 0.05, 'max': 0.2, 'offl': 0.01, 'offt': 0.5},
            'u2': {'stps': 5, 'cvp': 'LPF', 'const': 'rel', 'min': 5, 'max': 35, 'offl': 1, 'offt': 0.5}},
    'tvo': {'y1': {'init': 0, 'meas': True, 'sp': 10, 'offt': 0.5, 'samp_s': 1, 'samp_f': [], 'unc': 0.08},
            'y2': {'init': 0.01, 'meas': True, 'sp': 10, 'offt': 0.5, 'samp_s': 1, 'samp_f': [], 'unc': 0.08}},
    'tii': {'y10': {'min': 1, 'max': 5.5}},
    'tio': {},
    't_s': [0, 40], 't_r': 0.4 if QUICK else 0.16, 't_d': 0,
}

iden_opt = {'meth': 'SLSQP', 'ms': False, 'ob': 'WLS', 'maxit': 60 if QUICK else 500, 'tol': 1e-8,
            'var-cov': 'M', 'sens_m': 'central', 'eps': 1e-3,
            'c_plt': False, 'f_plt': True, 'plt_s': True, 'log': False}


def ptest(unc):
    """P-test: the probability (%) that each candidate is the right model, given the data."""
    return pd.Series({m: unc['results'][m]['P'] for m in models['can_m']}, name='P [%]').round(1)

In [ ]:
round_data = {}
expera(system, models, insilicos, {}, expr=1)
unc = uncert(parmest(system, models, iden_opt), system, models, iden_opt)   # BFF needs the covariances
save_rounds(1, unc['results'], 'preliminary', round_data, models, iden_opt, unc['obs'], system)
P_before = ptest(unc)
P_before

### One design per criterion

In [ ]:
designs = {}
for crit in (['BFF'] if QUICK else ['HR', 'BFF']):
    print(f'---------------- {crit}')
    des_opt = {'md_ob': crit, 'meth': 'PS', 'eps': 1e-3, 'plt': True,
               'itr': {'maxmd': 15 if QUICK else 100, 'tolmd': 1, 'pps': 30}}
    designs[crit] = mbdoe_md(des_opt, system, models, round=2, num_parallel_runs=1 if QUICK else 4)
pd.DataFrame({c: {'y10': d['tii'].get('y10'), 'criterion value': d['md_obj']} for c, d in designs.items()}).T

The two values are on different scales and cannot be compared directly; compare the
designed profiles in the plots. Maintenance shows when growth is slow (low substrate, long
residence), so both criteria look for such periods; BFF prefers those where the predictions are
also *certain*.

### Run the BFF design

In [ ]:
d = designs['BFF']
expera(system, models, insilicos, d, expr=2, swps=d['swps'])
unc = uncert(parmest(system, models, iden_opt), system, models, iden_opt)
save_rounds(2, unc['results'], 'MBDoE-MD (BFF)', round_data, models, iden_opt, unc['obs'], system)
pd.concat({'1 preliminary run': P_before, '+ 1 designed run': ptest(unc)}, axis=1)

**Try:** run the HR design instead and compare the P-test; add a second preliminary run
(smaller parameter uncertainty) and see the two designs become more alike.